<a href="https://colab.research.google.com/github/chelseanbr/aih-final-project/blob/main/notebooks/02-generate-mcqs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. Explore and sample the MedQuAD dataset
2. ***Generate ~3.5k medical multiple-choice questions (MCQs) using Phi-2***
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 2. Generate MCQs using Phi-2

In [1]:
import json
from tqdm import tqdm
from google.colab import files

# Upload JSONL file with question and answer fields
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving medquad_sampled.jsonl to medquad_sampled (4).jsonl


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "microsoft/phi-2"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Load and parse JSONL file
with open(file_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.float16).to(device)
model.eval()
print('Tokenizer and model loaded.')

In [5]:
# Load and parse JSONL file
with open(file_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.float16).to(device)
model.eval()

# Format QA into prompt
def format_prompt(question, answer):
    return (
        f"Convert the following medical Q&A into a multiple-choice question with one correct answer "
        f"and three plausible but incorrect options.\n\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n\n"
        f"Multiple-choice question:"
    )

# Batch-generate MCQs
BATCH_SIZE = 8

results = []
for i in tqdm(range(0, 100, BATCH_SIZE)):  # DEBUG
# for i in tqdm(range(0, len(qa_data), BATCH_SIZE)):
    batch = qa_data[i:i + BATCH_SIZE]
    prompts = [format_prompt(q["question"], q["answer"]) for q in batch]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    for prompt, completion in zip(prompts, decoded):
        results.append({"prompt": prompt, "completion": completion})

# Save to JSONL file
output_file = "generated_mcqs.jsonl"
with open(output_file, "w") as f:
    for item in results:
        json.dump(item, f)
        f.write("\n")

print(f"Generated MCQs saved to {output_file}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 38%|███▊      | 5/13 [01:32<02:28, 18.58s/it]


KeyboardInterrupt: 